In [32]:
import pandas as pd
import numpy as np

def extract_and_merge_missing_parquet_data(df_master, csv_path):
    temp_grid_load_weather = pd.read_csv(csv_path)
    
    cities = [
        'nashville', 'memphis', 'knoxville', 'chattanooga', 'clarksville', 
        'murfreesboro', 'franklin', 'johnson_city', 'jackson', 'hendersonville'
    ]
    
    # Extract Global Weighted Features
    weighted_cols = [
        'timestamp', 'weighted_temperature_2m', 'weighted_relative_humidity_2m', 
        'weighted_precipitation', 'weighted_cloud_cover', 
        'weighted_wind_speed_10m', 'weighted_shortwave_radiation'
    ]
    weighted_df = temp_grid_load_weather[weighted_cols].copy()
    weighted_df['timestamp'] = pd.to_datetime(weighted_df['timestamp'], utc=True).dt.tz_localize(None)
    
    # Reshape City-Level Humidity
    city_humidity_list = []
    for city in cities:
        city_col = f"{city}_relative_humidity_2m"
        if city_col in temp_grid_load_weather.columns:
            temp = temp_grid_load_weather[['timestamp', city_col]].copy()
            # Force absolute lowercase and spaces for guaranteed matching
            temp['city_name'] = city.replace('_', ' ').lower()
            temp = temp.rename(columns={city_col: 'city_relative_humidity'})
            city_humidity_list.append(temp)
    
    long_humidity_df = pd.concat(city_humidity_list, ignore_index=True)
    long_humidity_df['timestamp'] = pd.to_datetime(long_humidity_df['timestamp'], utc=True).dt.tz_localize(None)

    # Merge with Master
    df_master = df_master.merge(weighted_df, on='timestamp', how='left')
    df_master = df_master.merge(long_humidity_df, on=['timestamp', 'city_name'], how='left')
    
    return df_master


def get_master_data(use_live_db=False):
    if use_live_db:
        print("Connecting to PostgreSQL live database...")
        pass

    else:
        print("Loading local Parquet files...")
        base_path = '/workspaces/CECS-399-499/local_data/gold/'
        
        energy_feat = pd.read_parquet(f'{base_path}fact_energy_features_hourly.parquet')
        energy_load = pd.read_parquet(f'{base_path}fact_energy_load_hourly.parquet')
        time_dim = pd.read_parquet(f'{base_path}dim_time_hourly.parquet')
        city_dim = pd.read_parquet(f'{base_path}dim_city.parquet')
        weather_city = pd.read_parquet(f'{base_path}fact_weather_city_hourly.parquet')
        
        # Merge Grid Load & Features (TVA level)
        df = pd.merge(energy_feat, energy_load, on=['time_key', 'source_id'])
        
        # Merge Time Data
        df = pd.merge(df, time_dim, left_on='time_key', right_on='time_id')
        
        # Merge Weather Data ON TIME ONLY (Broadcasts 1 TVA row to 10 city rows)
        df = pd.merge(df, weather_city, on='time_key')
        
        # Merge City Data
        df = pd.merge(df, city_dim, left_on='city_key', right_on='city_id')
        
        # Create Naive Timestamp
        df['timestamp'] = pd.to_datetime(df['time_key'], utc=True).dt.tz_localize(None)
        
        # Force master city_name to lowercase to align with the extraction script
        if 'city_name' in df.columns:
            df['city_name'] = df['city_name'].str.replace('_', ' ').str.lower()
        
        # Merge Missing Parquet Data 
        missing_data_path = '/workspaces/CECS-399-499/local_data/silver/tn_weighted_weather_21_25.csv'
        df = extract_and_merge_missing_parquet_data(df, missing_data_path)
        
        # --- DATA CLEANUP ---
        # Drop redundant keys AND the broken coordinate columns since the model ignores them
        cols_to_drop = ['time_id', 'city_key', 'city_id', 'time_key', 'source_id', 'latitude', 'longitude']
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

        if 'is_weekend' in df.columns:
            df['is_weekend'] = df['is_weekend'].astype(int)
        
        df = df.sort_values(['city_name', 'timestamp']).reset_index(drop=True)
        
        return df

# Build the master dataframe
df_master = get_master_data(use_live_db=False)

# Quick validation of the fix
print("\n--- NEW MISSING VALUES ---")
print(df_master.isnull().sum())

Loading local Parquet files...

--- NEW MISSING VALUES ---
net_interchange_mwh              490
balance_error                    490
forecast_error_pct               890
demand_ramp_pct                  510
demand_rolling_mean               50
demand_rolling_std                70
demand_residual                  490
demand_zscore                    500
hour_sin                           0
hour_cos                           0
month_sin                          0
month_cos                          0
demand_forecast_mwh              680
actual_demand_mwh                490
net_gen_mwh                      490
timestamp                          0
hour_of_day                        0
day_of_week                        0
is_weekend                         0
month                              0
quarter                            0
year                               0
temperature                        0
precipitation                      0
cloud_cover                        0
wind_speed      

In [33]:
import pandas as pd

base_path = '/workspaces/CECS-399-499/local_data/gold/'

# Load raw energy parquets
energy_load = pd.read_parquet(f'{base_path}fact_energy_load_hourly.parquet')
energy_feat = pd.read_parquet(f'{base_path}fact_energy_features_hourly.parquet')

print("--- RAW FACT_ENERGY_LOAD_HOURLY MISSINGNESS ---")
# Filter to only show columns with missing data for cleaner output
load_missing = energy_load.isnull().sum()
print(load_missing[load_missing > 0])

print("\n--- RAW FACT_ENERGY_FEATURES_HOURLY MISSINGNESS ---")
feat_missing = energy_feat.isnull().sum()
print(feat_missing[feat_missing > 0])

print("\n--- QUICK VERIFICATION ---")
print(f"Raw actual_demand_mwh missing: {energy_load['actual_demand_mwh'].isnull().sum()}")
print(f"Master actual_demand_mwh missing: {energy_load['actual_demand_mwh'].isnull().sum() * 10} (Expected in df_master)")

--- RAW FACT_ENERGY_LOAD_HOURLY MISSINGNESS ---
demand_forecast_mwh    68
actual_demand_mwh      49
net_gen_mwh            49
dtype: int64

--- RAW FACT_ENERGY_FEATURES_HOURLY MISSINGNESS ---
net_interchange_mwh    49
balance_error          49
forecast_error_pct     89
demand_ramp_pct        51
demand_rolling_mean     5
demand_rolling_std      7
demand_residual        49
demand_zscore          50
dtype: int64

--- QUICK VERIFICATION ---
Raw actual_demand_mwh missing: 49
Master actual_demand_mwh missing: 490 (Expected in df_master)


In [40]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# 1. PRE-MODELING CLEANUP
# Drop remaining missing data as specified
initial_len = len(df_master)
df_master = df_master.dropna().reset_index(drop=True)
print(f"Dropped {initial_len - len(df_master)} rows containing missing data.")

# 2. FEATURE SELECTION
all_numeric = df_master.select_dtypes(include=[np.number]).columns.tolist()
exclude_from_fit = [
    'source_id', 'city_id', 'city_key', 'time_id', 'latitude', 
    'longitude', 'population', 'year', 'quarter', 'month', 
    'hour_of_day', 'day_of_week'
]
FEATURES = [f for f in all_numeric if f not in exclude_from_fit]

# 3. CHRONOLOGICAL SPLIT (60/40)
df_master['timestamp'] = pd.to_datetime(df_master['timestamp'])
df_master = df_master.sort_values('timestamp').reset_index(drop=True)

start_date = df_master['timestamp'].min()
end_date = df_master['timestamp'].max()
total_days = (end_date - start_date).days
split_point = start_date + pd.Timedelta(days=int(total_days * 0.6))

train_df = df_master[df_master['timestamp'] < split_point].copy()
test_df = df_master[df_master['timestamp'] >= split_point].copy()

# 4. MODEL TRAINING
model = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
model.fit(train_df[FEATURES])

train_df['anomaly'] = model.predict(train_df[FEATURES])
test_df['anomaly'] = model.predict(test_df[FEATURES])

# 5. ROBUST EXPLANATION LOGIC
# Uses Standard Deviation (sigma) to prevent ZeroDivisionError on cyclical/zero-mean features
normal_stats = train_df[train_df['anomaly'] == 1][FEATURES].agg(['mean', 'std']).T
normal_stats['std'] = normal_stats['std'].replace(0, 1e-9) 

def explain_anomaly(row):
    if row['anomaly'] == 1: 
        return "Normal"
    
    # Impact score = number of standard deviations from normal mean
    impact = (row[FEATURES] - normal_stats['mean']) / normal_stats['std']
    top_drivers = impact.abs().sort_values(ascending=False).head(2)
    
    d1_name, d1_val = top_drivers.index[0], impact[top_drivers.index[0]]
    d2_name, d2_val = top_drivers.index[1], impact[top_drivers.index[1]]
    
    return f"{d1_name} ({d1_val:+.1f}σ), {d2_name} ({d2_val:+.1f}σ)"

train_df['explanation'] = train_df.apply(explain_anomaly, axis=1)
test_df['explanation'] = test_df.apply(explain_anomaly, axis=1)

# 6. VALIDATION (Direct Timestamp/Date Membership Lookup)
def run_validation(df, label):
    outage_path = '/workspaces/CECS-399-499/local_data/gold/fact_outage_daily.parquet'
    outages = pd.read_parquet(outage_path)
    
    # Extract unique set of dates where outages occurred
    outage_dates = set(pd.to_datetime(outages['date']).dt.date.unique())
    
    # Extract date from hourly timestamp
    df['event_date'] = pd.to_datetime(df['timestamp']).dt.date
    
    # Identify anomalies
    anomalies = df[df['anomaly'] == -1].copy()
    
    # Validation: Is the anomaly date present in the outage dataset?
    anomalies['is_validated'] = anomalies['event_date'].apply(lambda x: x in outage_dates)
    
    # Metrics calculation
    validated_hits = anomalies[anomalies['is_validated'] == True]
    unique_dates_caught = validated_hits['event_date'].nunique()
    
    # Total unique outage dates existing within this split's timeframe
    relevant_outage_dates = [d for d in outage_dates if d in df['event_date'].unique()]
    total_outage_days = len(relevant_outage_dates)
    
    print(f"\n--- {label} VALIDATION ---")
    print(f"Timeframe:         {df['event_date'].min()} to {df['event_date'].max()}")
    print(f"Outage Days Caught: {unique_dates_caught} / {total_outage_days} ({unique_dates_caught/total_outage_days if total_outage_days > 0 else 0:.1%})")
    print(f"Total Anomalies:    {len(anomalies)}")
    print(f"Validated Hits:     {len(validated_hits)}")
    
    return validated_hits

# Execute
train_hits = run_validation(train_df, "TRAINING SPLIT")
test_hits = run_validation(test_df, "TESTING SPLIT")

if not test_hits.empty:
    print("\n--- VALIDATED TRACEBACK REPORT (TEST SET) ---")
    display(test_hits[['timestamp', 'city_name', 'explanation', 'actual_demand_mwh', 'balance_error']].head(15))

Dropped 0 rows containing missing data.

--- TRAINING SPLIT VALIDATION ---
Timeframe:         2021-01-01 to 2023-12-31
Outage Days Caught: 8 / 248 (3.2%)
Total Anomalies:    2620
Validated Hits:     16

--- TESTING SPLIT VALIDATION ---
Timeframe:         2023-12-31 to 2025-12-31
Outage Days Caught: 9 / 81 (11.1%)
Total Anomalies:    4218
Validated Hits:     784

--- VALIDATED TRACEBACK REPORT (TEST SET) ---


,timestamp,city_name,explanation,actual_demand_mwh,balance_error
351347,2025-01-06 18:00:00,johnson city,"wind_speed (+3.4σ), weighted_wind_speed_10m (+...",25964.0,3.0
351349,2025-01-06 18:00:00,memphis,"weighted_wind_speed_10m (+3.2σ), net_gen_mwh (...",25964.0,3.0
351350,2025-01-06 19:00:00,johnson city,"weighted_wind_speed_10m (+3.0σ), wind_speed (+...",26273.0,5.0
351362,2025-01-06 20:00:00,johnson city,"wind_speed (+3.0σ), weighted_wind_speed_10m (+...",26647.0,4.0
351363,2025-01-06 20:00:00,jackson,"wind_speed (+2.9σ), weighted_wind_speed_10m (+...",26647.0,4.0
351369,2025-01-06 20:00:00,memphis,"weighted_wind_speed_10m (+2.8σ), net_gen_mwh (...",26647.0,4.0
351370,2025-01-06 21:00:00,memphis,"net_gen_mwh (+2.9σ), weighted_wind_speed_10m (...",26888.0,3.0
351375,2025-01-06 21:00:00,johnson city,"wind_speed (+3.5σ), net_gen_mwh (+2.9σ)",26888.0,3.0
351382,2025-01-06 22:00:00,jackson,"net_gen_mwh (+3.0σ), wind_speed (+2.4σ)",27217.0,5.0
351390,2025-01-06 23:00:00,clarksville,"net_gen_mwh (+3.2σ), wind_speed (+2.0σ)",27743.0,3.0
